In [127]:
import numpy as np

class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.grad = None
        self.creator = None
        self.generation = 0
    def set_creator(self, creator):
        self.creator = creator
        self.generation = creator.generation + 1
    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = []
        set_seen = set()
        def add_func(f):
            set_seen.append(f)
            funcs.append(f)
            funcs.sort(key=lambda x:x.generation)
        add_func(self.creator)
        while funcs:
            f = funcs.pop()
            gys = [y.grad for y in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)
            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx
                else:
                    x.grad = x.grad + gx
                if x.creator is not None:
                    add_func(x.creator)
            #y = f.output
            #x = f.input
            #x.grad = f.backward(y.grad)
    def cleargrad(self):
        self.grad = None

In [128]:
class Function:
    def __call__(self, *inputs):
        # print(type(inputs[0][0]))
        xs = [x.data for x in inputs]
        # print(x)
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        # print('y:{}'.format(y))
        outputs = [Variable(as_array(y)) for y in ys]
        self.generation = max([x.generation for x in inputs])
        for output in outputs:
            output.set_creator(self)
        # print('output.data:{}'.format(output.data))
        # output.set_creator(self)
        self.inputs = inputs
        self.outputs = outputs
        
        return outputs if len(outputs) > 1 else outputs[0]
    def forward(self, x):
        raise NotImplementedError
    def backward(self, x):
        raise NotImplementedError
def as_array(x):
    if np.isscalar(x):
        x = np.array(x)
    return x
class Square(Function):
    def forward(self, x):
        return x**2
    def backward(self, gy):
        x = self.input[0].data
        gx = 2 * x * gy
        return gx
class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx
class Add(Function):
    def forward(self, x0, x1):
        y = x0+x1
        return (y,)
    def backward(self, gy):
        return gy, gy
def square(x):
    return Square()(x)
def exp(x):
    return Exp()(x)
def add(x0, x1):
    return Add()(x0, x1)

In [126]:
A = Square()
B = Exp()
C = Square()
D = Add()
x = Variable(np.array(1.))
a = A(x)
b = B(a)
c = C(a)
y = D(b,c)

assert y.creator == D
assert b.creator == B
assert c.creator == C
assert a.creator == A
assert D.inputs == (b,c)
assert D.outputs[0] == y
assert C.inputs == (a,)
assert C.outputs[0] == c

# print('generation,A{},B{},C{},D{}'.format(A.generation, B.generation, C.generation,D.generation))

funcs = [B,C,D,A]
print([f.generation for f in funcs])
funcs.sort(key=lambda x:x.generation)
print([f.generation for f in funcs])

[1, 1, 2, 0]
[0, 1, 1, 2]


In [77]:
x = Variable(np.array(1))
y = add(add(x,x),x)
y.backward()
print(y.data)
print(x.grad)

3
3


In [99]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(1.))
a = A(x)
b = B(a)
y = C(b)

assert y.creator == C
assert C.output == y
assert C.input == b
assert b.creator == B
assert B.output == b
assert B.input == a
assert a.creator == A
assert A.output == a
assert A.input == x


y.backward()
print(x.grad)

29.556224395722598


In [59]:
x = Variable(np.array(1))
y = Variable(np.array(1))
z = add(x,y)
z.backward()
print(z.data)
print(x.grad)

2
1


In [53]:
x = Variable(np.array(1))
y = Variable(np.array(1))
z = square(y)

print(type(z.data))
print(z.data)

<class 'numpy.ndarray'>
1


In [51]:
x0 = Variable(np.array(1.))
x1 = Variable(np.array(2.))
f = Add()
y = add(x0, x1)

print(y.data)


3.0


In [89]:
if not []:
    print('不空')

空


In [52]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(1.))
a = A(x)
b = B(a)
y = C(b)
y.grad = np.array(1)
print(y.grad)
b.grad = C.backward(y.grad)
a.grad = B.backward(b.grad)
x.grad = A.backward(a.grad)

print(y.data)
print(x.grad)

a:1.0,
b:2.718281828459045,
y:7.3890560989306495
1
C.input.data:2.718281828459045
b.grad:5.43656365691809
7.3890560989306495
29.556224395722598


In [29]:
x = Variable(np.array(0))
f = Exp()
y = f(x)
dy = numercial_diff(x, f)
print(dy)

1.0000000016668897
